# Lab03 — 02. Binary sentiment modeling với Hugging Face

**MODE="train"**, **DEVICE="cuda"**: train từ pretrained base trên Kaggle GPU; chạy từ trên xuống. Local dùng cùng notebook/module, chọn CPU hoặc CUDA phù hợp. Notebook bao gồm pretrained sentiment, baseline TF-IDF và DistilBERT regularized; huấn luyện **tối đa5 epoch với early stopping**.

| Phase | Nội dung | Module/file tham chiếu | Artifact trong outputs/ |
|---|---|---|---|
| 1 | Setup, environment và sanity tests | `src/notebook_runtime.py`, `tests/` | preflight log/XML |
| 2 | Pretrained sentiment / tokenization | `src/models/classifier.py`, `scripts/run_exercise1.py` | predictions, tokens/IDs/mask, comparison PNG |
| 3 | Architecture, optimizer, regularization | `config/frozen_trial.json`, `src/training/frozen.py` | config, optimizer groups, architecture |
| 4 | Baseline + fine-tuning | `scripts/run_frozen_trial.py`, `src/training/trainer.py` | epoch_history, learning_curves |
| 5 | Accuracy/loss và time/RAM/VRAM | `src/performance.py` | performance PNG/JSON, resource_history |
| 6 | Validation checkpoint / threshold sweep | `src/training/thresholds.py` | frozen policy, 19-point scan |
| 7 | Test và calibration | `src/training/metrics.py` | confusion, metrics bar, ROC/PR, reliability |
| 8 | Error analysis | `src/study.py`, `src/utils/plots.py` | samples, threshold_errors |
| 9 | Save/load, publish và ZIP | `src/utils/runs.py`, `scripts/publish_artifacts.py` | log, catalog, SHA256, review/checkpoint ZIP |

Dữ liệu và feature engineering được giải thích trong [notebook01](01_eda_preprocessing_feature_engineering.ipynb). Các bảng/ảnh bên dưới đọc artifact của chính phiên thực thi; không ghi trước accuracy hoặc selected epoch. MODE=view dùng outputs/latest_report.json sau khi đã có kết quả.

## 1. Setup và preflight

Kaggle: bật GPU và Internet. Python3.12 riêng tránh xung đột thư viện kernel. Local cần cài requirements trước khi mở notebook; chọn `DEVICE="cpu"` nếu chủ động dùng CPU. Không train full mạng neural trong bước kiểm thử: preflight dùng tiny fixtures, sanity overfit một minibatch và kiểm tra early stopping/threshold/reload/artifact.

Run All theo thứ tự để các biến session/config và helper luôn được định nghĩa.

In [ ]:
from pathlib import Path
import subprocess, sys, json
MODE = "train"
DEVICE = "cuda"
REPO_URL = "https://github.com/lacthui06/Lab-DL.git"
REPO_REF = "lab03"
start = Path.cwd().resolve()
REPO = next((p for p in (start,*start.parents) if (p/"labs/lab03/config/frozen_trial.json").is_file()), None)
if REPO is None and Path("/kaggle/working").is_dir():
    REPO = Path("/kaggle/working/Lab-DL")
    if not REPO.exists():
        subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO)],check=True)
assert REPO is not None, "Mở notebook trong repository hoặc trên Kaggle."
profile = json.loads((REPO/"labs/lab03/config/frozen_trial.json").read_text())["exercise2"]
assert profile["num_train_epochs"] == 5 and profile.get("early_stopping", {}).get("enabled"), "Cập nhật source branch lab03 cho đúng profile của notebook trước khi chạy."
sys.path.insert(0,str(REPO))
from labs.lab03.src.notebook_runtime import NotebookSession
session = NotebookSession(REPO, MODE, DEVICE)
assert "LAB03_SESSION_ID" in session.env, "Source repository chưa đồng bộ với notebook; cập nhật branch lab03 và mở session Kaggle mới."
session.setup()
from IPython.display import display


In [ ]:
session.preflight()

## 2. Exercise1 — Pretrained sentiment và tokenization

Dùng sentiment model đã fine-tune SST-2 cho inference. Pipeline và manual forward/softmax phải cho cùng nhãn/xác suất trong tolerance; xem tokens, input IDs và attention mask. Confidence là xác suất theo mô hình, không bảo đảm đúng trong mọi ngữ cảnh. Exercise2 dùng **DistilBERT base + classification head mới**, không dùng sentiment checkpoint SST-2 để fine-tune.

In [ ]:
session.exercise1()
display(session.table("exercise1_current", "tokenizer_breakdown.json"))
display(session.table("exercise1_current", "predictions.json", "manual"))
session.show("exercise1_visuals", ["tokenizer_table.png", "sentiment_predictions.png", "pipeline_manual_comparison.png"])


## 3. Architecture và cấu hình

Input IDs/mask `[B,L]` → WordPiece/position embedding →6 Transformer blocks → `[CLS]` vector → Linear768/ReLU → dropout0.4 → Linear2 logits. Hidden768, attention12heads, FFN3072. Train bằng cross-entropy trên logits; inference dùng softmax rồi threshold Positive.

Chọn DistilBERT để tận dụng pretrained representation với corpus nhỏ và ngân sách T4. Freeze embedding và4blocks đầu, chỉ học2blocks cuối +head. TF-IDF unigram/bigram + Logistic Regression là baseline đơn giản để kiểm tra Transformer có mang lại lợi ích không; không nối sparse features vào head.

Regularization: dropout encoder/attention0.2, head0.4; AdamW decay0.1 cho weights>=2D, không decay bias/LayerNorm; clipping1; warmup10% rồi linear decay. EncoderLR5e-6, headLR2e-5 vì head khởi tạo mới. Augmentation chỉ từ train, giới hạn25%, chặn trùng/câu có phủ định-tương phản và cân bằng variants.

Max5epoch, early stopping theo validation loss, patience2/min_delta0.001. Đây là thiết kế định trước cần kiểm chứng bằng kết quả, không bảo đảm tối ưu tuyệt đối hoặc không overfit.

In [ ]:
config = session.read(session.lab/"config/frozen_trial.json")
display(config["exercise2"])
display(config["exercise2"]["early_stopping"])


## 4. Baseline và fine-tuning tối đa5 epoch

Pipeline tải official8.530/1.066/1.066 splits, kiểm dữ liệu, tạo augmented train rồi fit TF-IDF baseline train-only. Chọn C∈{0.1,1,10} bằng validation Macro-F1. Transformer khởi tạo từ pretrained base, không resume một model đã fine-tune.

Mỗi epoch đo **full effective train và full validation trong eval mode**. Accuracy/loss end-epoch khác optimization loss chịu dropout lúc training. Early stopping chỉ đếm validation evaluations; evaluate(train) không làm tăng patience. Nếu dừng trước5 thì biểu đồ có đúng số điểm đã chạy. TrụcX là epoch; step để định danh optimizer update/checkpoint.

Checkpoint được chọn theo min validation loss thực tế. Min_delta điều khiển patience, không đổi tiêu chí checkpoint.

In [ ]:
session.train()
print("Run:", session.paths["model"].name)
display(session.table("model", "epoch_history.json"))
session.show("model", ["learning_curves.png"])
session.show("model", ["data_quality.png", "lengths_by_class.png", "augmentation_diagnostics.png"])


### Đọc đường học

Train loss giảm nhưng validation loss tăng thường cho thấy mô hình ngày càng tự tin vào lỗi hoặc học chi tiết riêng của train. Accuracy không đổi vẫn có thể đi cùng loss tăng. Gap nhỏ chưa đủ chứng minh tốt: cũng có thể underfit, phải so mức validation performance và baseline.

Đọc completed_epochs/stopped_early và selected_epoch từ artifact. Không điền thêm điểm, không đổi log hoặc chọn hyperparameter theo test.

## 5. Performance: accuracy/loss, time/sample, RAM–VRAM/sample

Bảng tách metric theo epoch khỏi tài nguyên. `train_ms_per_sample` đo train phase, không gồm end-epoch evaluation/save; epoch_pipeline có train + full train-eval +validation. Test timing là batch evaluation throughput, **không phải latency một request**.

RAM là parent-process RSS peak sampled20ms. VRAM là PyTorch allocated/reserved peak trên GPU được chọn. Peak chia nominal batch slots để biểu diễn bộ nhớ dùng chung theo batch, không phải bộ nhớ riêng của từng câu. Giá trị thiếu giữ null; CPU VRAM là N/A. Sơ đồ bên dưới đọc cấu trúc model.config đã ghi khi load model.

In [ ]:
performance = session.performance()
perf_table = session.table("performance", "performance_history.json")
display(perf_table[["epoch", "train_accuracy", "validation_accuracy", "train_loss", "validation_loss"]])
display(perf_table[["epoch", "train_ms_per_sample", "epoch_pipeline_ms_per_train_sample",
                    "ram_peak_rss_mib", "ram_mib_per_batch_slot", "vram_peak_allocated_mib",
                    "vram_peak_reserved_mib", "vram_mib_per_batch_slot"]])
display({k:performance[k] for k in ["test_eval_ms_per_sample", "test_eval_timing_scope", "memory_status", "source_device"]})
session.show("performance", ["performance.png", "model_architecture.png"])


## 6. Chọn checkpoint và quét threshold trên validation

Sau khi reload checkpoint min validation loss, quét19thresholds0.05–0.95 bước0.05 bằng validation Macro-F1. Hòa điểm thì chọn gần0.5 hơn, tiếp đến threshold nhỏ hơn. Chốt policy và hash weights trước khi test; không ép threshold khác0.5.

Precision/recall thường đánh đổi khi threshold tăng. Grid và tie-break đã định trước; không mở rộng grid chỉ vì nhìn thấy test result.

In [ ]:
display(session.table("model", "selection_summary.json"))
policy = session.read(session.paths["model"]/"decision_threshold.json")
display(policy["selected_metrics"])
display(session.table("model", "validation_threshold_scan.json"))
session.show("model", ["validation_threshold_curves.png"])


## 7. Đánh giá test và baseline

Áp dụng policy đã khóa cho toàn1.066test samples. So baseline, Transformer threshold0.5 và Transformer validation-selected threshold. Báo Accuracy, precision, recall, Binary/Macro-F1 và test loss; đọc confusion counts +row-normalized rates. So sánh này là của một lượt/seed, không mặc định Transformer luôn hơn baseline.

In [ ]:
display(session.table("model", "test_comparison.json"))
display(session.table("model", "test_runtime_metrics.json"))
session.show("model", ["metrics_bar.png", "confusion_matrix.png"])


### ROC/PR, calibration và uncertainty

ROC/PR khảo sát score ranking; reliability diagram, Brier và ECE khảo sát chất lượng confidence. Chúng là diagnostics sau khi khóa policy, không dùng test chọn threshold. Bootstrap interval chỉ mô tả fixed-model/test sampling, không bao gồm training-seed uncertainty. Accuracy cao vẫn có thể đi cùng confidence kém hoặc false positives/negatives mất cân đối.

In [ ]:
display(session.table("model", "probability_diagnostics.json"))
session.show("model", ["roc_pr_curves.png", "calibration_confidence.png"])


## 8. Phân tích lỗi định tính

Đọc câu đúng/sai, nhãn thật, nhãn dự đoán và confidence; chú ý phủ định, tương phản, châm biếm và implicit sentiment. Bảng lấy high-confidence examples theo nhóm đúng/sai và lớp nên **không phải mẫu ngẫu nhiên đại diện toàn test**. Giải thích ngôn ngữ là giả thuyết, không phải causal attribution.

`threshold_errors` lưu các câu sai dưới policy đã chọn. Đọc trade-off FP/FN trong test_comparison thay vì chỉ một metric tổng.

In [ ]:
session.show("model", ["sample_predictions.png"])
display(session.table("model", "threshold_errors.json").head(12))
display(session.table("model", "test_predictions.json").head(8))


## 9. Save/load, publish và tải artifact

Save model/tokenizer/policy, reload và so logits trên validation trong tolerance. Mỗi run lưu config, source hashes, environment, log, predictions, PNG/JSON và SHA256 manifest. Mỗi hình hiển thị trong notebook là cùng file PNG lưu trên disk.

Kết thúc training tự publish hình/bảng/log vào `outputs/published/`, nguồn ở `outputs/latest_report.json`. **Review ZIP** chứa artifact nhẹ/manifest/log; **checkpoint ZIP** riêng chứa weights để reload/evaluate/infer. Giữ checkpoint riêng, không đưa weights/ZIP lên Git.

Tải notebook của phiên Kaggle qua File → Download Notebook nếu cần lưu cell outputs; interactive notebook không tự nằm trên disk của training container.

In [ ]:
reload_check = session.read(session.paths["model"]/"reload_check.json")
display({k:v for k,v in reload_check.items() if k != "predictions"})
print("Published report:", session.lab/"outputs/published/index.md")
session.export()


### Câu hỏi bảo vệ

- Vì sao pretrained base thay vì học Transformer từ đầu? Tận dụng biểu diễn ngôn ngữ và giảm nhu cầu dữ liệu/compute.
- Vì sao có baseline? Để kiểm tra lợi ích thực tế của mô hình phức tạp.
- Freeze có tắt dropout không? Không; requires_grad=False ngăn cập nhật weights, eval mode mới tắt dropout.
- Vì sao có thể ít hơn5epoch? Early stopping theo validation đã đạt patience.
- Vì sao validation loss dùng chọn checkpoint còn Macro-F1 dùng threshold? Hai bước tối ưu hai quyết định khác nhau, cùng chỉ dùng validation.
- Có bảo đảm không overfit không? Không; báo mức gap, calibration và giới hạn dữ liệu/seed.
- RAM/VRAM per batch slot có phải bộ nhớ riêng một câu không? Không; có shared model memory và phụ thuộc batch/dynamic padding.
- Lấy hình làm report có cần rerun không? Không, sau khi train thành công các hình/bảng/log đã publish.